# DenseNet

Huang, Liu, Van Der Maaten, Weinberger, *Densely Connected Convolutional Networks*, CVPR 2017 ([arXiv:1608.06993](https://arxiv.org/abs/1608.06993)).

Inside a DenseBlock, layer `i`'s input is the concatenation of the block's original input plus every earlier layer's output in that block -- `in_channels + i*growth_rate` channels. Nothing computed in the block is discarded before the block ends. This notebook uses 3 small DenseBlocks (growth_rate=12), much shallower than the paper's 100+-layer CIFAR configs. See `model.py`.

Trains on real CIFAR-10.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from cnn_playground.data import load_cifar10
from cnn_playground.device import resolve_device
from cnn_playground.utils.seed import set_seed
from model import DenseNetModel

set_seed(0)
device = resolve_device('auto')
print('device:', device)

In [ ]:
train_ds = load_cifar10(train=True)
test_ds = load_cifar10(train=False)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
print(len(train_ds), len(test_ds))

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.numel()
    return correct / total

model = DenseNetModel().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

history = {'train_loss': [], 'test_acc': []}
epochs = 20
for epoch in range(epochs):
    model.train()
    last_loss = None
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt.zero_grad()
        loss = loss_fn(model(imgs), labels)
        loss.backward()
        opt.step()
        last_loss = loss.item()
    history['train_loss'].append(last_loss)
    history['test_acc'].append(evaluate(model, test_loader))

print(f"final test accuracy: {history['test_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history['train_loss']); axes[0].set_title('train loss'); axes[0].set_xlabel('epoch')
axes[1].plot(history['test_acc']); axes[1].set_title('test accuracy'); axes[1].set_xlabel('epoch')
fig.tight_layout()
plt.show()

In [ ]:
classes = train_ds.classes
imgs, labels = next(iter(test_loader))
imgs, labels = imgs[:6].to(device), labels[:6]
preds = model(imgs).argmax(dim=1).cpu()

mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3,1,1)

fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for i, ax in enumerate(axes):
    img = (imgs[i].cpu() * std + mean).clamp(0,1).permute(1,2,0)
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'pred:{classes[preds[i]]}\ntrue:{classes[labels[i]]}', fontsize=9)
fig.tight_layout()
plt.show()